In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

In [ ]:
# Load results from the Elephant paper for comparison
elephant_results = pd.read_csv('./uni_mergedTable_VUS-PR.csv')

# Window sizes used for each method in the Elephant paper
# See `Optimal_Uni_algo_HP_dict` in https://github.com/thedatumorg/TSB-AD/blob/main/TSB_AD/HP_list.py
win_sizes = {'AutoEncoder':100, 'CNN':50,
       'LSTMAD':100, 'TranAD':10, 'AnomalyTransformer':50, 'OmniAnomaly':5, 'USAD':100,
       'Donut':60, 'TimesNet':32, 'FITS':100, 'OFA':50}

In [ ]:
column_specs = {
    'Variant': 'object', 
    'correlation_var': 'float64',
    'correlation_last_3': 'float64', 
}

df = pd.DataFrame({col: pd.Series(dtype=dtype) for col, dtype in column_specs.items()})

for model in elephant_results.columns[1:-12]:

    if model in win_sizes:
        win_size = win_sizes[model]
    else:
        win_size = 64 # Default window size for models involving no window size

    path_baseline_1 = f'../results/baselines/var_{win_size}.csv'
    path_baseline_2 = '../results/baselines/last_3.csv'

    results_baseline_1 = pd.read_csv(path_baseline_1)['VUS-PR'].to_numpy().astype(np.float32)
    results_baseline_2 = pd.read_csv(path_baseline_2)['VUS-PR'].to_numpy().astype(np.float32)

    results_model = elephant_results[model].to_numpy().astype(np.float32)

    correlation_var, _ = stats.pearsonr(results_model, results_baseline_1, alternative='greater')
    correlation_last_3, _ = stats.pearsonr(results_model, results_baseline_2, alternative='greater')

    new_row_data = pd.Series({'Variant': '\\textbf{' + model + '}', 'correlation_var':correlation_var, 'correlation_last_3':correlation_last_3})
    new_row_df = new_row_data.to_frame().T
    df = pd.concat([df, new_row_df], ignore_index=True)

df


In [ ]:
format_dict = {col: '{:.3f}' for col in df.select_dtypes(include='object').columns}
del format_dict['Variant']

print(df.style.format(format_dict).hide().to_latex())